In [18]:
import os, subprocess, time, socket

# ── 1. Get Postgres password from Key Vault ──────────────────────────────────
vault = os.environ.get("KV_NAME")
if not vault:
    raise EnvironmentError(
        "KV_NAME is not set. Add this to your ~/.zshrc:\n"
        "  export KV_NAME=<your-vault-name>\n"
        "Then restart VS Code."
    )

result = subprocess.run(
    ["az", "keyvault", "secret", "show",
     "--vault-name", vault, "--name", "postgres-password",
     "--query", "value", "-o", "tsv"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Could not fetch secret from Key Vault:\n{result.stderr.strip()}")

os.environ["PGPASSWORD"] = result.stdout.strip()
print("✓ Password fetched from Key Vault")

# ── 2. Start kubectl port-forward (if not already running) ───────────────────
def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("localhost", port)) == 0

if port_open(5434):
    print("✓ Port 5434 already open — port-forward already running")
else:
    pf = subprocess.Popen(
        ["kubectl", "port-forward", "-n", "longevity", "svc/postgres-svc", "5434:5432"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    for _ in range(10):
        time.sleep(0.5)
        if port_open(5434):
            print(f"✓ Port-forward started (pid {pf.pid})")
            break
    else:
        pf.terminate()
        raise RuntimeError("Port-forward did not become ready. Is kubectl configured and the cluster reachable?")

✓ Password fetched from Key Vault
✓ Port-forward started (pid 73133)
